# Module 03-1: FakeLLM 구현 및 테스트 (솔루션)

## 학습 목표

1. **BaseChatModel 이해**: LangChain의 `BaseChatModel`을 상속하여 커스텀 LLM을 만드는 방법을 익힌다
2. **_generate 메서드 구현**: LLM의 핵심 메서드인 `_generate`를 구현하여 응답 생성 로직을 작성한다
3. **패턴-응답 매핑 설계**: 키워드 패턴과 응답을 딕셔너리로 매핑하는 구조를 이해한다
4. **ChatResult 구조 파악**: `AIMessage`, `ChatGeneration`, `ChatResult` 계층 구조를 파악한다
5. **FakeLLM 테스트**: 다양한 패턴 매칭 시나리오를 테스트하여 동작을 검증한다

---
> 참고 문서: `docs/part2-first-agent/03-jupyter-dev-environment.md`

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

from typing import Any, Optional
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatGeneration, ChatResult

print("환경 설정 완료")

## 실습 1: MyFakeLLM 클래스 구현 (솔루션)

In [ ]:
# 솔루션
class MyFakeLLM(BaseChatModel):
    responses: dict[str, str] = {}
    default_response: str = "MyFakeLLM 기본 응답입니다."

    @property
    def _llm_type(self) -> str:
        return "my-fake-llm"

    def _generate(self, messages: list[BaseMessage], stop=None, **kwargs) -> ChatResult:
        last_msg = messages[-1].content if messages else ""
        response = self.default_response
        for pattern, resp in self.responses.items():
            if pattern.lower() in last_msg.lower():
                response = resp
                break
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=response))])

print("MyFakeLLM 구현 완료")

In [ ]:
# 검증 셀
assert 'MyFakeLLM' in dir(), "MyFakeLLM 클래스가 정의되어야 합니다"
assert issubclass(MyFakeLLM, BaseChatModel), "MyFakeLLM은 BaseChatModel을 상속해야 합니다"

# 기본 응답 테스트
llm = MyFakeLLM()
result = llm.invoke("아무 질문")
assert hasattr(result, 'content'), "invoke 결과에 content 속성이 있어야 합니다"
assert result.content == "MyFakeLLM 기본 응답입니다.", f"기본 응답이 올바르지 않습니다: {result.content}"
print("✅ 실습 1 완료!")

## 실습 2: 패턴-응답 매핑 테스트 (솔루션)

In [ ]:
# 솔루션
llm_with_patterns = MyFakeLLM(
    responses={
        "요약": "이 문서의 핵심 내용은 LangGraph를 활용한 에이전트 구축입니다.",
        "분석": "분석 결과: 긍정적인 내용이 75%, 중립 25%로 구성되어 있습니다.",
        "번역": "Translation: This is the translated content."
    }
)

r_summary = llm_with_patterns.invoke("이 문서를 요약해주세요")
r_analyze = llm_with_patterns.invoke("이 텍스트를 분석해주세요")
r_translate = llm_with_patterns.invoke("이 내용을 번역해주세요")

print(f"요약 응답: {r_summary.content}")
print(f"분석 응답: {r_analyze.content}")
print(f"번역 응답: {r_translate.content}")
print("패턴-응답 테스트 완료")

In [ ]:
# 검증 셀
assert 'llm_with_patterns' in dir(), "llm_with_patterns 변수가 정의되어야 합니다"
assert len(llm_with_patterns.responses) >= 3, "최소 3개의 패턴-응답이 필요합니다"

r_summary = llm_with_patterns.invoke("이 문서를 요약해주세요")
assert "요약" in r_summary.content or "핵심" in r_summary.content or "LangGraph" in r_summary.content, \
    "요약 패턴의 응답이 올바르지 않습니다"

r_analyze = llm_with_patterns.invoke("이 텍스트를 분석해주세요")
assert "분석" in r_analyze.content, "분석 패턴의 응답에 '분석'이 포함되어야 합니다"

r_translate = llm_with_patterns.invoke("이 내용을 번역해주세요")
assert "Translation" in r_translate.content or "번역" in r_translate.content, \
    "번역 패턴의 응답이 올바르지 않습니다"

print("✅ 실습 2 완료!")

## 실습 3: common.fake_llm의 FakeLLM과 비교 (솔루션)

In [ ]:
# 솔루션
from common.fake_llm import FakeLLM
test_responses = {"안녕": "안녕하세요! 무엇을 도와드릴까요?"}
my_llm = MyFakeLLM(responses=test_responses)
common_llm = FakeLLM(responses=test_responses)

print(f"MyFakeLLM 응답: {my_llm.invoke('안녕하세요').content}")
print(f"FakeLLM 응답:   {common_llm.invoke('안녕하세요').content}")
print("비교 테스트 완료")

In [ ]:
# 검증 셀
from common.fake_llm import FakeLLM
test_resp = {"테스트": "테스트 응답입니다"}
my = MyFakeLLM(responses=test_resp)
common = FakeLLM(responses=test_resp)

assert my.invoke("테스트 질문").content == common.invoke("테스트 질문").content, \
    "MyFakeLLM과 FakeLLM의 응답이 동일해야 합니다"
print(f"MyFakeLLM 응답: {my.invoke('테스트 질문').content}")
print(f"FakeLLM 응답:   {common.invoke('테스트 질문').content}")
print("✅ 실습 3 완료!")

## 정리

### 오늘 배운 내용
- `BaseChatModel`을 상속하여 **커스텀 LLM** 구현
- `_generate` 메서드: `messages → 패턴 매칭 → ChatResult` 흐름
- `AIMessage → ChatGeneration → ChatResult` 계층 구조
- 패턴-응답 딕셔너리를 활용한 **결정론적 테스트 환경** 구성

### 다음 노트북 안내
**02_three_node_graph.ipynb** - FakeLLM을 활용하여 3노드 그래프를 구성하고 Mermaid로 시각화합니다.